# 08 — CBIS-DDSM Ensemble Evaluation

Evaluates the 4-model ensemble (DaViT-CC + DaViT-MLO + DaViT-ALL-CC + DaViT-ALL-MLO) on the CBIS-DDSM test set.

Expected result (paper): Accuracy = 83.33%, F1 = 82.78%

In [ ]:
import torch
from torch.utils.data import DataLoader

from multiview_davit.config import load_config
from multiview_davit.models.ensemble import MultiViewEnsemble
from multiview_davit.data.datasets import PairedViewDataset
from multiview_davit.data.transforms import build_inference_transform
from multiview_davit.evaluation.ensemble_eval import evaluate_ensemble, dump_predictions_csv
from multiview_davit.evaluation.confusion import plot_confusion_matrix
from multiview_davit.evaluation.stats import wilson_ci

ens_cfg  = load_config('configs/ensemble.yaml')
data_cfg = load_config('configs/data.yaml')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
ensemble = MultiViewEnsemble.from_config(ens_cfg, device=device)

test_ds = PairedViewDataset(
    data_cfg.cbis_ddsm.test_csv,
    transform=build_inference_transform(ens_cfg.inference.input_size)
)
test_loader = DataLoader(test_ds, batch_size=ens_cfg.inference.batch_size,
                          shuffle=False, num_workers=4)

In [ ]:
metrics = evaluate_ensemble(ensemble, test_loader, device=device, mode='cbis')

n = len(test_ds)
n_correct = int(round(metrics['accuracy'] * n))
lo, hi = wilson_ci(n_correct, n)

print(f"Accuracy:          {metrics['accuracy']*100:.2f}%  (95% CI: [{lo*100:.2f}%, {hi*100:.2f}%])")
print(f"Balanced Accuracy: {metrics['balanced_accuracy']*100:.2f}%")
print(f"F1 (weighted):     {metrics['f1']:.4f}")
print(f"AUC:               {metrics['auc']:.4f}")
print(f"Sensitivity:       {metrics['sensitivity']:.4f}")
print(f"Specificity:       {metrics['specificity']:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(metrics['y_true'], metrics['y_pred'])
fig = plot_confusion_matrix(cm, save_path='results/cbis_confusion.png')

dump_predictions_csv(metrics['y_true'], metrics['y_pred'], metrics['y_prob'],
                     save_path='results/cbis_predictions.csv')